# Week 8: precision, profiling and the NaN post-mortem

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

## 1. How coarse are 16-bit floats?

The gap between neighbouring representable numbers grows with the number itself. fp16 also stops at 65504.

In [ ]:
x = torch.logspace(-3, 6, 400, dtype=torch.float64)
for dtype in (torch.float16, torch.bfloat16):
    nxt = torch.nextafter(x.to(dtype), torch.tensor(float("inf"), dtype=dtype)).double()
    gap = (nxt - x.to(dtype).double()) / x
    plt.loglog(x, gap, label=str(dtype))
plt.axvline(65504, color="red", ls="--", label="fp16 max")
plt.xlabel("value")
plt.ylabel("relative gap to next float")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. The filter NaN, replayed

Your 30 September filter run: GradScaler starting at 4096, growth interval 2000, a corrupt event every ~1400
steps. Compare with events only every 5000 steps.

In [ ]:
from exercises.week08.ex01_precision import simulate_corrupt_events

for every in (5000, 1400):
    s = simulate_corrupt_events(40_000, every, growth_interval=2000, init_scale=4096.0)
    plt.semilogy(np.maximum(s.history, 1e-12), label=f"inf gradient every {every} steps")
plt.axhline(1, color="grey", ls=":")
plt.xlabel("optimiser step")
plt.ylabel("loss scale")
plt.legend()
plt.show()

## 3. Mixed precision on your GPU

Same tracker, same data, fp32 vs bf16 autocast: time per step and peak memory.

Do not expect bf16 to win here (on a 4070 Ti both columns come out about equal, even at `dim=1024`). One
toy event has ~200 hits, so each step is a few hundred tiny kernels plus the matcher on the CPU: the time
goes on launch overhead, not arithmetic. The memory is mostly weights, gradients and AdamW state, which
mixed precision keeps in float32; only the activations shrink, and with 200 tokens there are few of them.
Both gains grow with the number of tokens, which is why they matter for 60k-hit TrackML events.

In [ ]:
from exercises.week06.ex02_build_tracker import build_tracker
from minihep.data.dataset import ToyTrackingDataset

data_dir = ensure_toy_data(num_train=50)
fields = ["x", "y", "z", "r", "eta", "phi"]
ds = ToyTrackingDataset(str(data_dir / "train"), {"hit": fields}, {"hit": ["on_valid_particle"]}, event_max_num_particles=32)
inputs, targets = (({k: v.to(DEVICE) for k, v in d.items()}) for d in ds[0])
model = build_tracker(fields, dim=128, num_queries=32).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)


def step(dtype):
    with torch.autocast(DEVICE, dtype=dtype, enabled=dtype != torch.float32):
        _, _, losses = model.loss(model(inputs), targets)
        loss = sum(v for layer in losses.values() for task in layer.values() for v in task.values())
    opt.zero_grad()
    loss.backward()
    opt.step()


from exercises.week08.ex02_debugging_and_speed import peak_memory_mb, time_fn

for dtype in (torch.float32, torch.bfloat16):
    ms = time_fn(lambda: step(dtype), warmup=2, iters=5 if FAST else 20, device=DEVICE)
    mb = f"{peak_memory_mb(lambda: step(dtype)):7.1f} MB peak" if DEVICE == "cuda" else ""
    print(f"{dtype!s:15s} {ms:7.1f} ms/step  {mb}")

## 4. Where does the time go? (torch.profiler)

In [ ]:
from torch.profiler import ProfilerActivity, profile

activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if DEVICE == "cuda" else [])
with profile(activities=activities, record_shapes=False) as prof:
    for _ in range(3):
        step(torch.float32)
sort_key = "cuda_time_total" if DEVICE == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

Look for `linear_sum_assignment` / the matcher: on real TrackML events it dominated your training step (vault
note "Hungarian matching overhead"). On toy events it is small because the cost matrix is only 32 x ~10.

## 5. torch.compile

In [ ]:
import copy

eager = copy.deepcopy(model).eval()
compiled = copy.deepcopy(model).eval()
compiled.encoder.compile(dynamic=True)
with torch.no_grad():
    t_eager = time_fn(lambda: eager(inputs), warmup=2, iters=10, device=DEVICE)
    t_comp = time_fn(lambda: compiled(inputs), warmup=3, iters=10, device=DEVICE)  # the first calls compile
print(f"eager {t_eager:.2f} ms   compiled encoder {t_comp:.2f} ms")

## Your turn

1. Make the step NaN on purpose (multiply one input by 1e6 and cast inputs to `.half()` like hepattn's data
   loader), then find the culprit with `first_nonfinite_module`.
2. Train the tracker through the CLI with `--trainer.precision bf16-mixed` and `16-mixed`. Which needs a
   GradScaler? Find it in the checkpoint: `torch.load(ckpt)["MixedPrecision"]` (this is how you diagnosed
   the scale collapse).
3. Profile a hepattn TrackML step on Hypatia with `trainer.profiler` (commented out in the TrackML configs).